# Figure 5

In [1]:
options(warn=-1)

In [2]:
library_load <- suppressMessages(
    
    suppressWarnings(
        
        list(
        
            # Seurat 
            library(Seurat), 
                         
            # Data 
            library(tidyverse), 
            library(data.table), 
            library(reactable), 

            # Plotting 
            library(ComplexHeatmap), 
            library(patchwork), 
            library(cowplot), 
            library(ggrepel), 
            library(Nebulosa), 
            
            # Parallel 
            library(BiocParallel), 

            # Pyhton compatibility
            library(reticulate)

        )
    )
)

In [3]:
# Configure reticulate 
use_condaenv(condaenv="p.3.8.12-FD20220623HSCT", conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)

In [4]:
random_seed <- 42
set.seed(random_seed)

In [5]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20220623HSCT")

In [6]:
# Plotting Theme
source("plotting_global.R")
ggplot2::theme_set(theme_global_set(size_select=4)) # From project global source()
# showtext::showtext_auto() 

In [7]:
# Source scripts 
source("bin/seurat_qc.R")
source("script/figure/bin/dp_feature.R")
source("script/figure/bin/dea_vp.R")
source("script/figure/bin/dea_stats.R")
source("script/figure/bin/line_plot.R")

# Import Seurat Object

In [8]:
so <- readRDS("data/object/pp_subset.rds")

# CD4+ 

## Import AnnData as Seurat 

In [9]:
# Load python libraries
sc <- import("scanpy", convert=FALSE)

# Import adata 
adata_0 <- sc$read_h5ad("data/object/pseudotime/scvi_pt_0.h5ad")

In [10]:
# Prepare meta data 
meta_0 <- py_to_r(adata_0$obs)
meta_0 <- droplevels(meta_0)

In [11]:
# Prepare sparse matrix for Seurat object 
counts <- py_to_r(adata_0$X)
var_names <- py_to_r(adata_0$var_names$to_list())
obs_names <- py_to_r(adata_0$obs_names$to_list())

counts <- Matrix::t(counts)
rownames(counts) <- var_names
colnames(counts) <- obs_names

In [12]:
# Prepare UMAP 
umap <- py_to_r(adata_0$obsm["X_umap"])
colnames(umap) <- c("UMAP_1", "UMAP_2")
rownames(umap) <- obs_names

In [13]:
so_0 <- CreateSeuratObject(counts, meta.data=meta_0)

In [14]:
# Add meta data
umap <- umap[colnames(so_0), ]
so_0[["umap"]] <- CreateDimReducObject(embeddings=as.matrix(umap), key="UMAP_")

In [15]:
# Normalize data 
so_0 <- NormalizeData(so_0, normalization.method="LogNormalize", scale.factor=10^6)

In [16]:
# Normalize module score 
so_0$ms_ccr6_il7r <- ifelse(so_0$ms_ccr6<=0 | so_0$ms_il7r<=0, 0, so_0$ms_ccr6_il7r)
so_0$ms_ccr6_il7r=(so_0$ms_ccr6_il7r - min(so_0$ms_ccr6_il7r)) / (max(so_0$ms_ccr6_il7r) - min(so_0$ms_ccr6_il7r))

In [17]:
# Get module score class
so_0$class_ccr6_il7r <- ifelse(so_0$ms_ccr6_il7r > 0, "CCR6+ IL7R+", "CCR6\u002D IL7R\u002D") %>% factor(., levels=c("CCR6+ IL7R+", "CCR6\u002D IL7R\u002D"))

In [18]:
# Set NA variables
so_0$host <- ifelse(so_0$genotype_class=="Host", as.character(so_0$genotype_class), NA) 
so_0$donor <- ifelse(so_0$genotype_class=="Donor", as.character(so_0$genotype_class), NA) 
so_0$baseline <- ifelse(so_0$time_point=="Baseline", as.character(so_0$time_point), NA) 
so_0$tx <- ifelse(so_0$time_point=="Tx", as.character(so_0$time_point), NA) 
so_0$d14 <- ifelse(so_0$time_point=="D14", as.character(so_0$time_point), NA) 
so_0$d100 <- ifelse(so_0$time_point=="D100", as.character(so_0$time_point), NA) 
so_0$gvhd <- ifelse(so_0$time_point=="GVHD", as.character(so_0$time_point), NA)

In [19]:
# Density line data 
data <- cbind(so_0[["umap"]]@cell.embeddings, so_0@meta.data) 

In [20]:
fplot_1 <- fplot(so_0, reduction="umap", features=c("ms_ccr6_il7r"), slot="data", alpha=1, pt_size=0.5, size_select=4, viridis_option="A") + 
    ggtitle("CCR6 / IL7R") + 
    guides(color=guide_colourbar(title="MS", barwidth=unit(2, "mm"), barheight=unit(10, "mm"))) 

In [21]:
fplot_2 <- fplot(so_0, reduction="umap", features=c("CCR6"), slot="data", alpha=1, pt_size=0.5, size_select=4, viridis_option="A") + 
    ggtitle("CCR6") + 
    guides(color=guide_colourbar(title="log(CPM)", barwidth=unit(2, "mm"), barheight=unit(10, "mm"))) 

In [22]:
fplot_3 <- fplot(so_0, reduction="umap", features=c("IL7R"), slot="data", alpha=1, pt_size=0.5, size_select=4, viridis_option="A") + 
    ggtitle("IL7R") + 
    guides(color=guide_colourbar(title="log(CPM)", barwidth=unit(2, "mm"), barheight=unit(10, "mm"))) 

In [23]:
dplot_1 <- dplot(so_0, reduction="umap", group_by="host", alpha=1, pt_size=0.5, size_select=4, order=TRUE) +
    ggtitle("Host") +
    scale_color_manual(values=color$genotype_class, na.value="#D4D4D4") + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm")) + 
    stat_density_2d(data=data[data$genotype_class=="Host", ], aes(x=UMAP_1, y=UMAP_2), color="black", size=0.2) 

In [24]:
dplot_2 <- dplot(so_0, reduction="umap", group_by="donor", alpha=1, pt_size=0.5, size_select=4, order=TRUE) +
    ggtitle("Baseline") +
    scale_color_manual(values=color$genotype_class, na.value="#D4D4D4") + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm")) + 
    stat_density_2d(data=data[data$genotype_class=="Donor", ], aes(x=UMAP_1, y=UMAP_2), color="black", size=0.2) 

In [25]:
dplot_3 <- dplot(so_0, reduction="umap", group_by="baseline", alpha=1, pt_size=0.5, size_select=4, order=TRUE) +
    ggtitle("Baseline") +
    scale_color_manual(values=color$time_point, na.value="#D4D4D4") + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm")) + 
    stat_density_2d(data=data[data$time_point=="Baseline", ], aes(x=UMAP_1, y=UMAP_2), color="black", size=0.2) 

In [26]:
dplot_4 <- dplot(so_0, reduction="umap", group_by="tx", alpha=1, pt_size=0.5, size_select=4, order=TRUE) +
    ggtitle("Tx") +
    scale_color_manual(values=color$time_point, na.value="#D4D4D4") + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm")) + 
    stat_density_2d(data=data[data$time_point=="Tx", ], aes(x=UMAP_1, y=UMAP_2), color="black", size=0.2) 

In [27]:
dplot_5 <- dplot(so_0, reduction="umap", group_by="d14", alpha=1, pt_size=0.5, size_select=4, order=TRUE) +
    ggtitle("D14") +
    scale_color_manual(values=color$time_point, na.value="#D4D4D4") + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm")) + 
    stat_density_2d(data=data[data$time_point=="D14", ], aes(x=UMAP_1, y=UMAP_2), color="black", size=0.2) 

In [28]:
dplot_6 <- dplot(so_0, reduction="umap", group_by="d100", alpha=1, pt_size=0.5, size_select=4, order=TRUE) +
    ggtitle("D100") +
    scale_color_manual(values=color$time_point, na.value="#D4D4D4") + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm")) + 
    stat_density_2d(data=data[data$time_point=="D100", ], aes(x=UMAP_1, y=UMAP_2), color="black", size=0.2) 

In [29]:
dplot_7 <- dplot(so_0, reduction="umap", group_by="gvhd", alpha=1, pt_size=0.5, size_select=4, order=TRUE) + 
    ggtitle("GVHD") +
    scale_color_manual(values=color$time_point, na.value="#D4D4D4") + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm")) + 
    stat_density_2d(data=data[data$time_point=="GVHD", ], aes(x=UMAP_1, y=UMAP_2), color="black", size=0.2) 

In [30]:
pdf("result/figures/figure_5/umap_cd4pos_ccr6_il7r.pdf", width=9, height=2)

fplot_1 + fplot_2 + fplot_3 + patchwork::plot_layout(ncol=3)

dev.off()

png 
  2

In [31]:
pdf("result/figures/figure_5/umap_cd4pos_genotype_density.pdf", width=9, height=2)

dplot_1 + dplot_2 + patchwork::plot_layout(ncol=2)

dev.off()

png 
  2

In [32]:
pdf("result/figures/figure_5/umap_cd4pos_time_point_density.pdf", width=15, height=2)

dplot_3 + dplot_4 + dplot_5 + dplot_6 + dplot_7 + patchwork::plot_layout(ncol=5)

dev.off()

png 
  2

## Re-population of skin PAGA cluster 

In [33]:
bp_1 <- ggplot(so_0@meta.data, aes(x=time_point, fill=class_ccr6_il7r)) + 
    ggtitle("T cells CD4+") + xlab("") + ylab("Ratio") + 
    facet_grid(~hto_demux_class) + 
    geom_bar(position='fill', color="black", size=0.1) + 
    scale_fill_manual(values=c("CCR6+ IL7R+"="#F24949", "CCR6\u002D IL7R\u002D"="#097560")) +
    guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.1, default.unit="cm", title="Class", reverse=FALSE)) + 
    
    theme(
        
        axis.text.x=element_text(angle=90, vjust=0.5, hjust=1)
        
    ) 

In [34]:
pdf("result/figures/figure_5/bp_cd4pos_time_point_ccr6_il7r_class_ratio.pdf", width=3, height=2)

bp_1

dev.off()

png 
  2

## Host donor enrichment 

In [35]:
data <- so_0@meta.data %>% 
    dplyr::mutate(class_ccr6_il7r=gsub("\207A", "pos", as.character(class_ccr6_il7r))) %>% 
    dplyr::select(patient_id, time_point, hto_demux_class, class_ccr6_il7r, genotype_class) %>% droplevels() %>% 
    dplyr::group_by(patient_id, time_point, hto_demux_class) %>% dplyr::mutate(n_cells_total=n()) %>% 
    dplyr::filter(is.na(class_ccr6_il7r)==FALSE) %>% 
    dplyr::group_by(genotype_class, class_ccr6_il7r, .add=TRUE) %>% dplyr::mutate(n_cells_gc=n()) %>% 
    dplyr::group_by(n_cells_total, .add=TRUE) %>% group_by(across(setdiff(group_vars(.), 'genotype_class'))) %>% distinct() %>% 
    tidyr::complete(., genotype_class, fill=list(n_cells_gc=0)) %>% ungroup() %>% 
    dplyr::mutate(cpm_cells_gc=10^3*n_cells_gc/n_cells_total) %>% as.data.frame()

In [36]:
data <- dplyr::group_by(data, time_point, class_ccr6_il7r, genotype_class, hto_demux_class) %>% 
    dplyr::mutate(mean=mean(cpm_cells_gc), sd=sd(cpm_cells_gc), n=length(unique(patient_id)), sem=sd/sqrt(n), ymin=ifelse(mean-sem<0, 0, mean-sem), ymax=mean+sem) %>%
    dplyr::select(-patient_id, -n_cells_total, -n_cells_gc, -cpm_cells_gc) %>% dplyr::distinct()

In [37]:
lp <- lp_time_point(data, title="T cells CD4+", ylab="10^3*' x cells / total CD4+ cells'", log_breaks=TRUE) + facet_grid(hto_demux_class~class_ccr6_il7r)

In [38]:
pdf("result/figures/figure_5/lp_cd4pos_ccr6_il7r_genotype_class.pdf", width=3, height=2)

lp

dev.off()

png 
  2

# CD8

## Import AnnData as Seurat 

In [39]:
# Load python libraries
sc <- import("scanpy", convert=FALSE)

# Import adata 
adata_1 <- sc$read_h5ad("data/object/pseudotime/scvi_pt_1.h5ad")

In [40]:
# Prepare meta data 
meta_1 <- py_to_r(adata_1$obs)
meta_1 <- droplevels(meta_1)

In [41]:
# Prepare sparse matrix for Seurat object 
counts <- py_to_r(adata_1$X)
var_names <- py_to_r(adata_1$var_names$to_list())
obs_names <- py_to_r(adata_1$obs_names$to_list())

counts <- Matrix::t(counts)
rownames(counts) <- var_names
colnames(counts) <- obs_names

In [42]:
# Prepare UMAP 
umap <- py_to_r(adata_1$obsm["X_umap"])
colnames(umap) <- c("UMAP_1", "UMAP_2")
rownames(umap) <- obs_names

In [43]:
so_1 <- CreateSeuratObject(counts, meta.data=meta_1)

In [44]:
# Add meta data
umap <- umap[colnames(so_1), ]
so_1[["umap"]] <- CreateDimReducObject(embeddings=as.matrix(umap), key="UMAP_")

In [45]:
# Normalize data 
so_1 <- NormalizeData(so_1, normalization.method="LogNormalize", scale.factor=10^6)

In [46]:
# Normalize module score 
so_1$ms_ccr6_il7r <- ifelse(so_1$ms_ccr6<=0 | so_1$ms_il7r<=0, 0, so_1$ms_ccr6_il7r)
so_1$ms_ccr6_il7r=(so_1$ms_ccr6_il7r - min(so_1$ms_ccr6_il7r)) / (max(so_1$ms_ccr6_il7r) - min(so_1$ms_ccr6_il7r))

In [47]:
# Get module score class
so_1$class_ccr6_il7r <- ifelse(so_1$ms_ccr6_il7r > 0, "CCR6+ IL7R+", "CCR6\u002D IL7R\u002D") %>% factor(., levels=c("CCR6+ IL7R+", "CCR6\u002D IL7R\u002D"))

In [48]:
# Set NA variables
so_1$host <- ifelse(so_1$genotype_class=="Host", as.character(so_1$genotype_class), NA) 
so_1$donor <- ifelse(so_1$genotype_class=="Donor", as.character(so_1$genotype_class), NA) 
so_1$baseline <- ifelse(so_1$time_point=="Baseline", as.character(so_1$time_point), NA) 
so_1$tx <- ifelse(so_1$time_point=="Tx", as.character(so_1$time_point), NA) 
so_1$d14 <- ifelse(so_1$time_point=="D14", as.character(so_1$time_point), NA) 
so_1$d100 <- ifelse(so_1$time_point=="D100", as.character(so_1$time_point), NA) 
so_1$gvhd <- ifelse(so_1$time_point=="GVHD", as.character(so_1$time_point), NA)

In [49]:
# Density line data 
data <- cbind(so_1[["umap"]]@cell.embeddings, so_1@meta.data) 

In [50]:
fplot_2 <- fplot(so_1, reduction="umap", features=c("CCR6"), slot="data", alpha=1, pt_size=0.5, size_select=4, viridis_option="A") + 
    ggtitle("CCR6") + guides(color=guide_colourbar(title="log(CPM)", barwidth=unit(2, "mm"), barheight=unit(10, "mm"))) 

In [51]:
fplot_3 <- fplot(so_1, reduction="umap", features=c("IL7R"), slot="data", alpha=1, pt_size=0.5, size_select=4, viridis_option="A") + 
    ggtitle("IL7R") + guides(color=guide_colourbar(title="log(CPM)", barwidth=unit(2, "mm"), barheight=unit(10, "mm"))) 

In [52]:
fplot_1 <- fplot(so_1, reduction="umap", features=c("ms_ccr6_il7r"), slot="data", alpha=1, pt_size=0.5, size_select=4, viridis_option="A") + 
    ggtitle("CCR6 / IL7R") + guides(color=guide_colourbar(title="MS", barwidth=unit(2, "mm"), barheight=unit(10, "mm"))) 

In [53]:
dplot_1 <- dplot(so_1, reduction="umap", group_by="host", alpha=1, pt_size=0.5, size_select=4, order=TRUE) +
    ggtitle("Host") +
    scale_color_manual(values=color$genotype_class, na.value="#D4D4D4") + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm")) + 
    stat_density_2d(data=data[data$genotype_class=="Host", ], aes(x=UMAP_1, y=UMAP_2), color="black", size=0.2) 

In [54]:
dplot_2 <- dplot(so_1, reduction="umap", group_by="donor", alpha=1, pt_size=0.5, size_select=4, order=TRUE) +
    ggtitle("Baseline") +
    scale_color_manual(values=color$genotype_class, na.value="#D4D4D4") +
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm")) + 
    stat_density_2d(data=data[data$genotype_class=="Donor", ], aes(x=UMAP_1, y=UMAP_2), color="black", size=0.2) 

In [55]:
dplot_3 <- dplot(so_1, reduction="umap", group_by="baseline", alpha=1, pt_size=0.5, size_select=4, order=TRUE) +
    ggtitle("Baseline") +
    scale_color_manual(values=color$time_point, na.value="#D4D4D4") + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm")) + 
    stat_density_2d(data=data[data$time_point=="Baseline", ], aes(x=UMAP_1, y=UMAP_2), color="black", size=0.2) 

In [56]:
dplot_4 <- dplot(so_1, reduction="umap", group_by="tx", alpha=1, pt_size=0.5, size_select=4, order=TRUE) +
    ggtitle("Tx") +
    scale_color_manual(values=color$time_point, na.value="#D4D4D4") + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm")) + 
    stat_density_2d(data=data[data$time_point=="Tx", ], aes(x=UMAP_1, y=UMAP_2), color="black", size=0.2) 

In [57]:
dplot_5 <- dplot(so_1, reduction="umap", group_by="d14", alpha=1, pt_size=0.5, size_select=4, order=TRUE) +
    ggtitle("D14") +
    scale_color_manual(values=color$time_point, na.value="#D4D4D4") + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm")) + 
    stat_density_2d(data=data[data$time_point=="D14", ], aes(x=UMAP_1, y=UMAP_2), color="black", size=0.2) 

In [58]:
dplot_6 <- dplot(so_1, reduction="umap", group_by="d100", alpha=1, pt_size=0.5, size_select=4, order=TRUE) +
    ggtitle("D100") +
    scale_color_manual(values=color$time_point, na.value="#D4D4D4") + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm")) + 
    stat_density_2d(data=data[data$time_point=="D100", ], aes(x=UMAP_1, y=UMAP_2), color="black", size=0.2) 

In [59]:
dplot_7 <- dplot(so_1, reduction="umap", group_by="gvhd", alpha=1, pt_size=0.5, size_select=4, order=TRUE) + 
    ggtitle("GVHD") +
    scale_color_manual(values=color$time_point, na.value="#D4D4D4") + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm")) + 
    stat_density_2d(data=data[data$time_point=="GVHD", ], aes(x=UMAP_1, y=UMAP_2), color="black", size=0.2) 

In [60]:
pdf("result/figures/figure_5/umap_cd8pos_ccr6_il7r.pdf", width=9, height=2)

fplot_1 + fplot_2 + fplot_3 + patchwork::plot_layout(ncol=3)

dev.off()

png 
  2

In [61]:
pdf("result/figures/figure_5/umap_cd8pos_genotype_density.pdf", width=9, height=2)

dplot_1 + dplot_2 + patchwork::plot_layout(ncol=2)

dev.off()

png 
  2

In [62]:
pdf("result/figures/figure_5/umap_cd8pos_time_point_density.pdf", width=15, height=2)

dplot_3 + dplot_4 + dplot_5 + dplot_6 + dplot_7 + patchwork::plot_layout(ncol=5)

dev.off()

png 
  2

## Re-population of skin PAGA cluster 

In [63]:
bp_1 <- ggplot(so_1@meta.data, aes(x=time_point, fill=class_ccr6_il7r)) + 
    ggtitle("T cells CD8+") + xlab("") + ylab("Ratio") + 
    facet_grid(~hto_demux_class) + 
    geom_bar(position='fill', color="black", size=0.1) + 
    scale_fill_manual(values=c("CCR6+ IL7R+"="#F24949", "CCR6\u002D IL7R\u002D"="#097560")) + 
    guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.1, default.unit="cm", title="Class", reverse=FALSE)) + 
    theme(axis.text.x=element_text(angle=90, vjust=0.5, hjust=1)) 

In [64]:
pdf("result/figures/figure_5/bp_cd8pos_time_point_ccr6_il7r_class_ratio.pdf", width=3, height=2)

bp_1

dev.off()

png 
  2

## Host donor enrichment 

In [65]:
data <- so_1@meta.data %>% 
    dplyr::mutate(class_ccr6_il7r=gsub("\207A", "pos", as.character(class_ccr6_il7r))) %>% 
    dplyr::select(patient_id, time_point, hto_demux_class, class_ccr6_il7r, genotype_class) %>% droplevels() %>% 
    dplyr::group_by(patient_id, time_point, hto_demux_class) %>% dplyr::mutate(n_cells_total=n()) %>% 
    dplyr::filter(is.na(class_ccr6_il7r)==FALSE) %>% 
    dplyr::group_by(genotype_class, class_ccr6_il7r, .add=TRUE) %>% dplyr::mutate(n_cells_gc=n()) %>% 
    dplyr::group_by(n_cells_total, .add=TRUE) %>% group_by(across(setdiff(group_vars(.), 'genotype_class'))) %>% distinct() %>% 
    tidyr::complete(., genotype_class, fill=list(n_cells_gc=0)) %>% ungroup() %>% 
    dplyr::mutate(cpm_cells_gc=10^3*n_cells_gc/n_cells_total) %>% as.data.frame()

In [66]:
data <- dplyr::group_by(data, time_point, class_ccr6_il7r, genotype_class, hto_demux_class) %>% 
    dplyr::mutate(mean=mean(cpm_cells_gc), sd=sd(cpm_cells_gc), n=length(unique(patient_id)), sem=sd/sqrt(n), ymin=ifelse(mean-sem<0, 0, mean-sem), ymax=mean+sem) %>%
    dplyr::select(-patient_id, -n_cells_total, -n_cells_gc, -cpm_cells_gc) %>% dplyr::distinct() %>% 
    dplyr::group_by(hto_demux_class, class_ccr6_il7r, genotype_class) %>% 
    tidyr::complete(., time_point, fill=list(mean=0, sd=NA, n=0, sem=NA, ymin=NA, ymax=NA)) %>% 
    dplyr::ungroup()

In [67]:
lp <- lp_time_point(data, title="T cells CD8+", ylab="10^3*' x cells / total CD8+ cells'", log_breaks=TRUE) + facet_grid(hto_demux_class~class_ccr6_il7r)

In [68]:
pdf("result/figures/figure_5/lp_cd8pos_ccr6_il7r_genotype_class.pdf", width=3, height=2)

lp

dev.off()

png 
  2

# Gene regulatory network analysis 

In [69]:
tf <- read.csv("script/figure/figure_5/data/figure_5_rss_score_selected.csv")

In [70]:
color_ccr6_il7r <- c("CCR6\u002D IL7R\u002D"="#F24949", "CCR6+ IL7R+"="#097560")

In [71]:
so <- readRDS("data/object/pp_subset.rds")

In [72]:
meta_0 <- so_0@meta.data[, c("cell_type_leiden_subset", "leiden")]
meta_1 <- so_1@meta.data[, c("cell_type_leiden_subset", "leiden")]

In [73]:
leiden_0 <- meta_0[meta_0$leiden %in% c("0", "14", "3", "8"), "leiden", drop=FALSE]
leiden_1 <- meta_1[meta_1$leiden %in% c("1", "2", "4", "8"), "leiden", drop=FALSE]

In [74]:
leiden <- rbind(leiden_0, leiden_1)
colnames(leiden) <- "leiden_paga"

In [75]:
so <- AddMetaData(so, leiden)

In [76]:
so$cell_type_leiden_subset <- ifelse(!is.na(so$leiden_paga) & so$cell_type_leiden_subset!="Cycling", paste0(so$cell_type_leiden_subset, paste0("*'", so$leiden_paga, "'")), as.character(so$cell_type_leiden_subset))

In [77]:
cell_type_all <- unique(so$cell_type_leiden_subset)

In [78]:
tcell_c_dp <- "T[\'c, memory/effector\']*\'4\'"
tcell_c_dn <- grep("c, memory/effector", cell_type_all, value=TRUE)[grep("c, memory/effector", cell_type_all, value=TRUE)!=tcell_c_dp]
    
tcell_h_dp <- "T[\'h, memory/effector\']*\'0\'"
tcell_h_dn <- grep("h, memory/effector", cell_type_all, value=TRUE)[grep("h, memory/effector", cell_type_all, value=TRUE)!=tcell_h_dp]
    
so$cell_type_leiden_subset[so$cell_type_leiden_subset %in% tcell_c_dp] <- "T['c, CCR6+ IL7R+']"
so$cell_type_leiden_subset[so$cell_type_leiden_subset %in% tcell_c_dn] <- "T['c, CCR6- IL7R-']"
so$cell_type_leiden_subset[so$cell_type_leiden_subset %in% tcell_h_dp] <- "T['h, CCR6+ IL7R+']"
so$cell_type_leiden_subset[so$cell_type_leiden_subset %in% tcell_h_dn] <- "T['h, CCR6- IL7R-']"

## Baseline - Skin 

In [79]:
meta <- so@meta.data
meta <- meta[meta$hto_demux_class=="Skin" & meta$time_point=="Baseline" & meta$cell_type_leiden_subset %in% c("T['c, CCR6- IL7R-']", "T['c, CCR6+ IL7R+']", "T['h, CCR6- IL7R-']", "T['h, CCR6+ IL7R+']"), ]

In [80]:
auc_mtx <- read.csv("result/scenic/lymphocyte/auc_mtx.csv", row.names=1)
colnames(auc_mtx) <- gsub("\\.", "", colnames(auc_mtx))

In [81]:
meta <- meta[intersect(rownames(meta), rownames(auc_mtx)), ]
auc_mtx <- auc_mtx[intersect(rownames(meta), rownames(auc_mtx)), ]

In [82]:
# Load python libraries
pysenic_rss <- import("pyscenic.rss", convert=FALSE)
pd <- import("pandas", convert=FALSE)

In [83]:
rss <- pysenic_rss$regulon_specificity_scores(auc_mtx, pd$Series(r_to_py(meta$cell_type_leiden_subset))) %>% py_to_r()

In [84]:
rss <- rss %>% tibble::rownames_to_column("cell_type_leiden_subset")

In [85]:
mat <- reshape2::melt(rss, id.vars=c("cell_type_leiden_subset"), variable.name="module", value.name="rss") %>% 
    
    dplyr::arrange(cell_type_leiden_subset, desc(rss)) %>% 
    dplyr::group_by(cell_type_leiden_subset) %>%
    dplyr::mutate(rank=1:n()) %>% 
    dplyr::filter(cell_type_leiden_subset %in% tf$cell_type_leiden_subset) %>% 
    dplyr::mutate(label=ifelse(paste0(cell_type_leiden_subset, module) %in% paste0(tf$cell_type_leiden_subset, tf$module), as.character(module), NA)) %>% 
    dplyr::mutate(label_rank=ifelse(rank %in% 1:15, as.character(module), NA))

In [86]:
mat_1 <- mat[mat$cell_type_leiden_subset %in% c("T['h, CCR6- IL7R-']", "T['h, CCR6+ IL7R+']"), ] %>% dplyr::mutate(cell_type_leiden_subset=ifelse(cell_type_leiden_subset=="T['h, CCR6+ IL7R+']", "CCR6+ IL7R+", "CCR6\u002D IL7R\u002D") %>% factor(., levels=c("CCR6\u002D IL7R\u002D", "CCR6+ IL7R+")))
mat_2 <- mat[mat$cell_type_leiden_subset %in% c("T['c, CCR6- IL7R-']", "T['c, CCR6+ IL7R+']"), ] %>% dplyr::mutate(cell_type_leiden_subset=ifelse(cell_type_leiden_subset=="T['c, CCR6+ IL7R+']", "CCR6+ IL7R+", "CCR6\u002D IL7R\u002D") %>% factor(., levels=c("CCR6\u002D IL7R\u002D", "CCR6+ IL7R+")))

In [87]:
dp_rss_top_label_1 <- ggplot(mat_1, aes(x=rank, y=rss, label=label_rank, color=cell_type_leiden_subset)) + 
    geom_point(size=1) + 
    xlab("") + ylab("RSS") + ggtitle("T cells CD4+ (Baseline)") + 
    ylim(c(min(mat$rss), max(mat$rss) + 0.1)) + 
    facet_grid(~cell_type_leiden_subset, scales="free_y") +
    scale_color_manual(values=color_ccr6_il7r, labels=names(color_ccr6_il7r), name="Cell type") + 
    scale_x_continuous(breaks=seq(0, floor(max(mat$rank)/100)*100, 100), labels=seq(0, floor(max(mat$rank)/100)*100, 100)) + 
    geom_text_repel(segment.color="black", force=10, force_pull=1, max.overlaps=getOption("ggrepel.max.overlaps", default=100), size=2, alpha=1, guide="none", segment.size=0.05, color="black", min.segment.length=0) + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm"))

dp_rss_top_label_2 <- ggplot(mat_2, aes(x=rank, y=rss, label=label_rank, color=cell_type_leiden_subset)) + 
    geom_point(size=1) + 
    xlab("") + ylab("RSS") + ggtitle("T cells CD8+ (Baseline)") + 
    ylim(c(min(mat$rss), max(mat$rss) + 0.1)) + 
    facet_grid(~cell_type_leiden_subset, scales="free_y") +
    scale_color_manual(values=color_ccr6_il7r, labels=names(color_ccr6_il7r), name="Cell type") + 
    scale_x_continuous(breaks=seq(0, floor(max(mat$rank)/100)*100, 100), labels=seq(0, floor(max(mat$rank)/100)*100, 100)) + 
    geom_text_repel(segment.color="black", force=10, force_pull=1, max.overlaps=getOption("ggrepel.max.overlaps", default=100), size=2, alpha=1, guide="none", segment.size=0.05, color="black", min.segment.length=0) + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm"))

In [88]:
pdf("result/figures/figure_5/rss_cd4pos_ccr6_il7r_top.pdf", width=3, height=2)

dp_rss_top_label_1

dev.off()

png 
  2

In [89]:
pdf("result/figures/figure_5/rss_cd8pos_ccr6_il7r_top.pdf", width=3, height=2)

dp_rss_top_label_2

dev.off()

png 
  2

In [90]:
dp_rss_top_label_1 <- ggplot(mat_1, aes(x=rank, y=rss, label=label, color=cell_type_leiden_subset)) + 
    geom_point(size=1) + 
    xlab("") + ylab("RSS") + ggtitle("T cells CD4+ (Baseline)") + 
    ylim(c(min(mat$rss), max(mat$rss) + 0.1)) + 
    facet_grid(~cell_type_leiden_subset, scales="free_y") +
    scale_color_manual(values=color_ccr6_il7r, labels=names(color_ccr6_il7r), name="Cell type") + 
    scale_x_continuous(breaks=seq(0, floor(max(mat$rank)/100)*100, 100), labels=seq(0, floor(max(mat$rank)/100)*100, 100)) + 
    geom_text_repel(segment.color="black", force=10, force_pull=1, max.overlaps=getOption("ggrepel.max.overlaps", default=100), size=2, alpha=1, guide="none", segment.size=0.05, color="black", min.segment.length=0) + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm"))

dp_rss_top_label_2 <- ggplot(mat_2, aes(x=rank, y=rss, label=label, color=cell_type_leiden_subset)) + 
    geom_point(size=1) + 
    xlab("") + ylab("RSS") + ggtitle("T cells CD8+ (Baseline)") + 
    ylim(c(min(mat$rss), max(mat$rss) + 0.1)) + 
    facet_grid(~cell_type_leiden_subset, scales="free_y") +
    scale_color_manual(values=color_ccr6_il7r, labels=names(color_ccr6_il7r), name="Cell type") + 
    scale_x_continuous(breaks=seq(0, floor(max(mat$rank)/100)*100, 100), labels=seq(0, floor(max(mat$rank)/100)*100, 100)) + 
    geom_text_repel(segment.color="black", force=10, force_pull=1, max.overlaps=getOption("ggrepel.max.overlaps", default=100), size=2, alpha=1, guide="none", segment.size=0.05, color="black", min.segment.length=0) + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm"))

In [91]:
pdf("result/figures/figure_5/rss_cd4pos_ccr6_il7r_selected.pdf", width=3, height=2)

dp_rss_top_label_1

dev.off()

png 
  2

In [92]:
pdf("result/figures/figure_5/rss_cd8pos_ccr6_il7r_selected.pdf", width=3, height=2)

dp_rss_top_label_2

dev.off()

png 
  2

# DEA CCR6 IL7R cells 

In [93]:
feature_select <- function(so, cnt_min, cell_min, grouping_var, random_effect_var, feature_group_min, verbose=TRUE) {
    
    # Select Seurat components 
    meta <- so@meta.data
    cnt <- GetAssayData(so, assay="RNA", slot="counts")
    
    # Prepare data for split 
    cnt <- t(as.matrix(cnt))
    cnt <- as.data.frame(cnt)
    
    # Split expression matrix by grouping variable into list
    cnt <- split(cnt, f=paste(so[[grouping_var, drop=TRUE]], so[[random_effect_var, drop=TRUE]]))
    
    # Check within group expression 
    cnt <- lapply(cnt, function(x) {colSums(x>=cnt_min)>=cell_min})
    cnt <- do.call(rbind, cnt)
    
    # Check across group expression 
    cnt_check <- colSums(cnt)>=feature_group_min
    
    # Get genes 
    genes <- colnames(cnt)[cnt_check]

    # Create Seurat object with gene subset 
    cnt <- GetAssayData(so, assay="RNA", slot="counts")
    cnt <- cnt[genes, , drop=FALSE]

    so <- CreateSeuratObject(counts=cnt, meta=meta)
    
    if(verbose) {message(paste("Final number of genes for testing", nrow(so)))}

    return(so)
    
}

In [94]:
voom_lm_fit <- function(so, grouping_var, random_effect_var, sample_weights) {

    # Get Counts 
    cnt <- GetAssayData(so, assay="RNA", slot="counts")
        
    # Prepare count data for split 
    cnt <- as.matrix(cnt)

    # Get grouping variables from cnt matrix
    grouping_var_vec <- so[[grouping_var, drop=TRUE]] # Used for design matrix
    random_effect_var_vec <- so[[random_effect_var, drop=TRUE]] # Used as blocking variable
    
    grouping_var_vec <- as.character(grouping_var_vec) 
    random_effect_var_vec <- as.character(random_effect_var_vec)
    
    design <- stats::model.matrix(~0 + grouping_var_vec + random_effect_var_vec)
    colnames(design) <- gsub("grouping_var_vec|random_effect_var_vec", "", colnames(design))
    
    # Run voomLmFit
    fit <- edgeR::voomLmFit(
        
        counts=cnt, 
        design=design, 
        block=NULL, 
        normalize.method="none", 
        lib.size=NULL, 
        sample.weights=sample_weights, 
        plot=FALSE

    )
    
    return(fit)
    
}

In [95]:
ebayes <- function(fit, so, grouping_var, grouping_var_vec) {
    
    contrasts_vec <- paste0(grouping_var_vec[1], "-", grouping_var_vec[2])
    contrasts <- limma::makeContrasts(contrasts=contrasts_vec, levels=fit$design)
    contrasts_fit <- limma::contrasts.fit(fit, contrasts=contrasts)
    
    # eBayes fit 
    efit <- limma::eBayes(contrasts_fit)

    # Get result table
    result_df <- limma::topTable(efit, sort.by="P", n=Inf, p.value=1, lfc=0, coef=1)
    
    # Convert results to Seurat format 
    colnames(result_df)[1] <- "avg_log2FC"
    colnames(result_df)[4] <- "p_value"
    colnames(result_df)[5] <- "p_val_adj"
            
    # Add percentage expression 
    cnt <- GetAssayData(so, assay="RNA", slot="counts")
    cnt_bool <- cnt[rownames(result_df), ] > 0

    result_df$pct_1<-rowSums(cnt_bool[, so[[grouping_var, drop=TRUE]]==grouping_var_vec[1]])/sum(so[[grouping_var, drop=TRUE]]==grouping_var_vec[1])
    result_df$pct_2<-rowSums(cnt_bool[, so[[grouping_var, drop=TRUE]]==grouping_var_vec[2]])/sum(so[[grouping_var, drop=TRUE]]==grouping_var_vec[2])
    
    return(result_df)
    
}

In [96]:
grouping_var <- "cell_type_leiden_subset_make_names"
random_effect_var <- "patient_id"
grouping_var_vec_1 <- c("T['h, CCR6pos IL7Rpos']", "T['h, CCR6neg IL7Rneg']") %>% make.names()
grouping_var_vec_2 <- c("T['c, CCR6pos IL7Rpos']", "T['c, CCR6neg IL7Rneg']") %>% make.names()
sample_weights <- FALSE

cnt_min=1
cell_min=1
feature_group_min=3

p_adj_thr <- 0.05

so$cell_type_leiden_subset_make_names <- so$cell_type_leiden_subset
so$cell_type_leiden_subset_make_names <- gsub("\\+", "pos", so$cell_type_leiden_subset_make_names)
so$cell_type_leiden_subset_make_names <- gsub("\\-", "neg", so$cell_type_leiden_subset_make_names)
so$cell_type_leiden_subset_make_names <- so$cell_type_leiden_subset_make_names %>% make.names()

so_1 <- subset(so, subset=cell_type_leiden_subset_make_names %in% grouping_var_vec_1 & time_point=="Baseline" & hto_demux_class=="Skin")
so_2 <- subset(so, subset=cell_type_leiden_subset_make_names %in% grouping_var_vec_2 & time_point=="Baseline" & hto_demux_class=="Skin")

In [97]:
so_1 <- feature_select(so_1, cnt_min, cell_min, grouping_var, random_effect_var, feature_group_min, verbose=TRUE)
so_2 <- feature_select(so_2, cnt_min, cell_min, grouping_var, random_effect_var, feature_group_min, verbose=TRUE)

Final number of genes for testing 8346

Final number of genes for testing 8767



In [98]:
fit_1 <- voom_lm_fit(so_1, grouping_var=grouping_var, random_effect_var=random_effect_var, sample_weights=sample_weights)
fit_2 <- voom_lm_fit(so_2, grouping_var=grouping_var, random_effect_var=random_effect_var, sample_weights=sample_weights)

In [99]:
result_1 <- ebayes(fit_1, so_1, grouping_var=grouping_var, grouping_var_vec=grouping_var_vec_1)
result_2 <- ebayes(fit_2, so_2, grouping_var=grouping_var, grouping_var_vec=grouping_var_vec_2)

In [100]:
label_1 <- read.csv("script/figure/figure_5/data/figure_5_dea_th_ccr6_il7r_pos_vs_neg_skin_selected.csv")$gene
label_2 <- read.csv("script/figure/figure_5/data/figure_5_dea_tc_ccr6_il7r_pos_vs_neg_skin_selected.csv")$gene

In [101]:
vp_1 <- dea_vp(result_1, title="T cells CD4+ (Baseline)", color_neg=color_ccr6_il7r[1], color_pos=color_ccr6_il7r[2], log2fc_thr=0, p_adj_thr=p_adj_thr, top_label=50, parse_title=FALSE, label=label_1) + 
    theme(legend.position="right") + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm"))
vp_2 <- dea_vp(result_2, title="T cells CD8+ (Baseline)", color_neg=color_ccr6_il7r[1], color_pos=color_ccr6_il7r[2], log2fc_thr=0, p_adj_thr=p_adj_thr, top_label=50, parse_title=FALSE, label=label_2) + 
    theme(legend.position="right") + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm"))

In [102]:
pdf("result/figures/figure_5/vp_dea_cd4pos_ccr6_il7r.pdf", width=3, height=2)

vp_1

dev.off()

png 
  2

In [103]:
pdf("result/figures/figure_5/vp_dea_cd8pos_ccr6_il7r.pdf", width=3, height=2)

vp_2

dev.off()

png 
  2